# CPSA 01 — build the half-bust YOLO-pose dataset

Inputs: the public COCO 2017 images dataset attached on Kaggle (any dataset
whose tree contains `train2017/` and `val2017/`, e.g. `awsaf49/coco-2017-dataset`).
The COCO-WholeBody v1.0 annotations are downloaded from the official Google
Drive links (https://github.com/jin-s13/COCO-WholeBody) with `gdown`; enable
Internet for this notebook. If the download is blocked, attach a dataset with
the two JSON files instead: the cell below uses them when present.

Set `COMMIT_SHA` below to the exact repository commit for this run. Outputs go
to `/kaggle/working/halfbust62` and are re-published as a dataset version
before the session ends. Set `SCHEMA = "coco17"` for Option 1. No GPU needed.


In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "REPLACE_WITH_COMMIT_SHA"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

# COCO-WholeBody v1.0 annotations (official Google Drive file ids).
WHOLEBODY_IDS = {
    "train": "1thErEToRbmM9uLNi1JXXfOsaS5VK2FXf",
    "val": "1N6VgwKnj8DeyGXCvp1eYgNbRmw6jdfrb",
}
ANN_DIR = Path("/kaggle/tmp/coco_wholebody")  # outside /kaggle/working: not published
ANN_DIR.mkdir(parents=True, exist_ok=True)


def find_attached(split):
    """Return an attached coco_wholebody_<split>_v1.0.json, if any."""
    name = f"coco_wholebody_{split}_v1.0.json"
    hits = sorted(Path("/kaggle/input").rglob(name))
    return hits[0] if hits else None


WHOLEBODY = {}
for split, file_id in WHOLEBODY_IDS.items():
    path = find_attached(split)
    if path is None:
        path = ANN_DIR / f"coco_wholebody_{split}_v1.0.json"
        if not path.is_file():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            subprocess.run(
                [sys.executable, "-m", "gdown", file_id, "-O", str(path)], check=True
            )
    with path.open() as handle:
        head = json.load(handle)
    assert head["info"]["description"] == "COCO-WholeBody", path
    print(split, path, len(head["images"]), "images", len(head["annotations"]), "annotations")
    del head
    WHOLEBODY[split] = path


def find_coco_root():
    """Return the attached directory that holds train2017/ and val2017/."""
    for candidate in sorted(Path("/kaggle/input").rglob("val2017")):
        root = candidate.parent
        if (root / "train2017").is_dir():
            return root
    raise FileNotFoundError("attach a COCO 2017 images dataset (train2017/ and val2017/)")


COCO_ROOT = find_coco_root()
print("COCO root:", COCO_ROOT)


In [ ]:
from cpsa_ml.datasets.halfbust import BuildConfig, build_halfbust_dataset

OUT = Path("/kaggle/working/halfbust62")
SCHEMA = "halfbust62"
OCCLUSION_P = 0.3
SEED = 0

for split in ("train", "val"):
    report = build_halfbust_dataset(
        BuildConfig(
            coco_root=COCO_ROOT,
            wholebody_json=WHOLEBODY[split],
            split=split,
            schema=SCHEMA,
            out=OUT,
            occlusion_p=OCCLUSION_P,
            seed=SEED,
        )
    )
    print(split, report.to_dict())


In [ ]:
print((OUT / "data.yaml").read_text())
print((OUT / "stats.json").read_text())

In [ ]:
# Re-publish /kaggle/working/halfbust62 as a Kaggle dataset version.
print("dataset root:", OUT)